In [ ]:
import os

DATA_DIR = "/kaggle/input/datasets/satwikshreshth01/bihar-kosi-flood-sar-datase/Kosi_FloodMapping_Dataset"

files = {
    "tile_001": "Bihar_Kosi_SAR_Stack-0000000000-0000000000-001.tif",
    "tile_02": "Bihar_Kosi_SAR_Stack-0000000000-0000013568.tif",
    "tile_03": "Bihar_Kosi_SAR_Stack-0000013568-0000000000.tif",
    "tile_04": "Bihar_Kosi_SAR_Stack-0000013568-0000013568.tif",
    "label": "Bihar_Kosi_Flood_Label.tif",
}

for name, fname in files.items():
    path = os.path.join(DATA_DIR, fname)
    exists = os.path.exists(path)
    size_gb = os.path.getsize(path) / 1e9 if exists else 0
    print(f"{name:10s} | exists={exists} | size={size_gb:.2f} GB | {fname}")

In [ ]:
import rasterio
import pandas as pd

audit = {}
for name, fname in files.items():
    path = os.path.join(DATA_DIR, fname)
    with rasterio.open(path) as src:
        audit[name] = {
            "width": src.width,
            "height": src.height,
            "band_count": src.count,
            "dtype": src.dtypes[0],
            "crs": src.crs,
            "transform": src.transform,
            "nodata": src.nodata,
            "bounds": src.bounds,
        }

df = pd.DataFrame(audit).T
df

In [ ]:
with rasterio.open(os.path.join(DATA_DIR, files["tile_001"])) as src:
    print("Band descriptions:", src.descriptions)
    print("Tags:", src.tags())
    for i in range(1, src.count + 1):
        print(f"Band {i} tags:", src.tags(i))

In [ ]:
# Aim: identify the actual fill/nodata value used at tile edges, since the
# file header reports nodata=None despite likely padding regions. This reads
# a small windowed sample from each tile's border rather than the full array,
# to keep memory use minimal.

import numpy as np

def sample_border_values(path, window_size=200):
    with rasterio.open(path) as src:
        window = rasterio.windows.Window(0, 0, min(window_size, src.width), min(window_size, src.height))
        sample = src.read(1, window=window)
    return sample

for name, fname in files.items():
    path = os.path.join(DATA_DIR, fname)
    sample = sample_border_values(path)
    print(f"{name:10s} | min={np.nanmin(sample):.4f} | max={np.nanmax(sample):.4f} | "
          f"unique_low_vals={np.unique(sample)[:5]} | has_nan={np.isnan(sample).any()}")

In [ ]:
# Aim: verify whether tile_03 and tile_04 contain NaN fill values anywhere,
# since the previous check only sampled the top-left corner. This checks all
# four corners of each tile using small windowed reads, keeping memory use
# minimal regardless of file size.

def sample_corners(path, window_size=200):
    with rasterio.open(path) as src:
        w, h = src.width, src.height
        ws = min(window_size, w)
        hs = min(window_size, h)
        corners = {
            "top_left": rasterio.windows.Window(0, 0, ws, hs),
            "top_right": rasterio.windows.Window(max(0, w - ws), 0, ws, hs),
            "bottom_left": rasterio.windows.Window(0, max(0, h - hs), ws, hs),
            "bottom_right": rasterio.windows.Window(max(0, w - ws), max(0, h - hs), ws, hs),
        }
        results = {}
        for corner_name, window in corners.items():
            sample = src.read(1, window=window)
            results[corner_name] = np.isnan(sample).any()
    return results

for name, fname in files.items():
    path = os.path.join(DATA_DIR, fname)
    corner_nan = sample_corners(path)
    print(f"{name:10s} | {corner_nan}")

In [ ]:
# Aim: visualize the no-data footprint shape across the full extent of each
# tile using a decimated read, to confirm whether the NaN region is a
# diagonal swath edge, a rectangular border, or something else. Using
# out_shape here means GDAL downsamples during the read itself, so the full
# resolution array is never loaded into memory.

import matplotlib.pyplot as plt

def read_downsampled(path, band=1, out_size=400):
    with rasterio.open(path) as src:
        scale = min(out_size / src.width, out_size / src.height)
        out_shape = (max(1, int(src.height * scale)), max(1, int(src.width * scale)))
        data = src.read(band, out_shape=out_shape, resampling=rasterio.enums.Resampling.average)
    return data

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
tile_names = list(files.keys())
for ax, name in zip(axes, tile_names):
    path = os.path.join(DATA_DIR, files[name])
    thumb = read_downsampled(path)
    ax.imshow(np.isnan(thumb), cmap="gray")
    ax.set_title(f"{name} (NaN mask)")
    ax.axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/phase0_nodata_footprint.png", dpi=100)
plt.show()

In [ ]:
# Aim: compute NaN fraction per tile using block-wise reads to quantify invalid area before mosaicking

def compute_nan_fraction_blockwise(path, band=1, block_size=1024):
    total_pixels = 0
    nan_pixels = 0
    with rasterio.open(path) as src:
        for row_off in range(0, src.height, block_size):
            row_size = min(block_size, src.height - row_off)
            for col_off in range(0, src.width, block_size):
                col_size = min(block_size, src.width - col_off)
                window = rasterio.windows.Window(col_off, row_off, col_size, row_size)
                block = src.read(band, window=window)
                total_pixels += block.size
                nan_pixels += np.isnan(block).sum()
    return nan_pixels, total_pixels

for name, fname in files.items():
    path = os.path.join(DATA_DIR, fname)
    nan_count, total_count = compute_nan_fraction_blockwise(path)
    pct = 100 * nan_count / total_count
    print(f"{name:10s} | nan_pixels={nan_count:,} | total_pixels={total_count:,} | nan_pct={pct:.2f}%")

In [ ]:
# Aim: compute flood vs non-flood pixel ratio in the label using block-wise reads to inform loss weighting later

def compute_class_balance_blockwise(path, band=1, block_size=1024):
    flood_pixels = 0
    valid_pixels = 0
    with rasterio.open(path) as src:
        for row_off in range(0, src.height, block_size):
            row_size = min(block_size, src.height - row_off)
            for col_off in range(0, src.width, block_size):
                col_size = min(block_size, src.width - col_off)
                window = rasterio.windows.Window(col_off, row_off, col_size, row_size)
                block = src.read(band, window=window)
                valid_mask = ~np.isnan(block)
                valid_pixels += valid_mask.sum()
                flood_pixels += (block[valid_mask] == 1).sum()
    return flood_pixels, valid_pixels

label_path = os.path.join(DATA_DIR, files["label"])
flood_count, valid_count = compute_class_balance_blockwise(label_path)
flood_pct = 100 * flood_count / valid_count
print(f"flood_pixels={flood_count:,} | valid_pixels={valid_count:,} | flood_pct={flood_pct:.2f}%")

In [ ]:
# Aim: save all Phase 0 audit results to disk as a summary artifact for reference in later notebooks

import json

audit_summary = {
    "tile_metadata": {k: {kk: str(vv) for kk, vv in v.items()} for k, v in audit.items()},
    "nodata_check": {
        "tile_001_nan_pct": 1.94,
        "tile_02_nan_pct": 5.96,
        "tile_03_nan_pct": 1.71,
        "tile_04_nan_pct": 0.76,
        "label_nan_pct": 0.16,
    },
    "class_balance": {
        "flood_pixels": int(flood_count),
        "valid_pixels": int(valid_count),
        "flood_pct": float(flood_pct),
    },
    "bands": ["VV_post", "VH_post", "VV_pre", "VH_pre", "log_ratio", "elevation"],
    "mosaic_dimensions": {"width": 16699, "height": 14495},
}

with open("/kaggle/working/phase0_audit_summary.json", "w") as f:
    json.dump(audit_summary, f, indent=2, default=str)

print("Audit summary saved to /kaggle/working/phase0_audit_summary.json")

In [ ]:
# Aim: define paths and confirm output budget before writing any mosaicked or patched data to disk

import os
import rasterio
import numpy as np

DATA_DIR = "/kaggle/input/datasets/satwikshreshth01/bihar-kosi-flood-sar-datase/Kosi_FloodMapping_Dataset"
OUTPUT_DIR = "/kaggle/working/patches"
os.makedirs(OUTPUT_DIR, exist_ok=True)

files = {
    "tile_001": "Bihar_Kosi_SAR_Stack-0000000000-0000000000-001.tif",
    "tile_02": "Bihar_Kosi_SAR_Stack-0000000000-0000013568.tif",
    "tile_03": "Bihar_Kosi_SAR_Stack-0000013568-0000000000.tif",
    "tile_04": "Bihar_Kosi_SAR_Stack-0000013568-0000013568.tif",
    "label": "Bihar_Kosi_Flood_Label.tif",
}

MOSAIC_WIDTH = 16699
MOSAIC_HEIGHT = 14495
BAND_COUNT = 6

full_res_gb = (MOSAIC_WIDTH * MOSAIC_HEIGHT * BAND_COUNT * 4) / 1e9
print(f"Estimated full-resolution SAR stack size at float32: {full_res_gb:.2f} GB")
print(f"Estimated label size at float32: {(MOSAIC_WIDTH * MOSAIC_HEIGHT * 4) / 1e9:.2f} GB")

In [ ]:
# Aim: build a virtual mosaic (VRT) from the four SAR tiles without duplicating pixel data on disk

from osgeo import gdal

tile_paths = [
    os.path.join(DATA_DIR, files["tile_001"]),
    os.path.join(DATA_DIR, files["tile_02"]),
    os.path.join(DATA_DIR, files["tile_03"]),
    os.path.join(DATA_DIR, files["tile_04"]),
]

vrt_path = "/kaggle/working/sar_mosaic.vrt"
vrt_options = gdal.BuildVRTOptions(resampleAlg="nearest", addAlpha=False)
vrt = gdal.BuildVRT(vrt_path, tile_paths, options=vrt_options)
vrt = None  # flush and close

vrt_size_kb = os.path.getsize(vrt_path) / 1e3
print(f"VRT created at {vrt_path}, size={vrt_size_kb:.2f} KB")

with rasterio.open(vrt_path) as src:
    print(f"VRT dimensions: width={src.width}, height={src.height}, bands={src.count}")

In [ ]:
# Aim: estimate total patch count and storage size before writing any patches to disk

PATCH_SIZE = 256
STRIDE = 256  # no overlap for baseline; can revisit later if more training data is needed

n_rows = (MOSAIC_HEIGHT + STRIDE - 1) // STRIDE
n_cols = (MOSAIC_WIDTH + STRIDE - 1) // STRIDE
total_patches = n_rows * n_cols

sar_patch_bytes_f16 = PATCH_SIZE * PATCH_SIZE * BAND_COUNT * 2  # float16
label_patch_bytes_u8 = PATCH_SIZE * PATCH_SIZE * 1  # uint8, since label is binary

est_total_gb = total_patches * (sar_patch_bytes_f16 + label_patch_bytes_u8) / 1e9

print(f"Patch grid: {n_rows} rows x {n_cols} cols = {total_patches} total patches")
print(f"Per-patch size: SAR={sar_patch_bytes_f16/1e6:.3f} MB (float16), label={label_patch_bytes_u8/1e6:.3f} MB (uint8)")
print(f"Estimated total storage if all patches kept: {est_total_gb:.2f} GB")

In [ ]:
# Aim: extract patches from the VRT and label, keeping only patches above a minimum valid-pixel threshold

import numpy as np
import rasterio
from rasterio.windows import Window

VALID_THRESHOLD = 0.7  # minimum fraction of non-NaN pixels required to keep a patch
manifest = []

sar_src = rasterio.open(vrt_path)
label_src = rasterio.open(os.path.join(DATA_DIR, files["label"]))

patch_id = 0
for row in range(n_rows):
    row_off = row * STRIDE
    row_size = min(PATCH_SIZE, MOSAIC_HEIGHT - row_off)
    for col in range(n_cols):
        col_off = col * STRIDE
        col_size = min(PATCH_SIZE, MOSAIC_WIDTH - col_off)

        window = Window(col_off, row_off, col_size, row_size)
        sar_patch = sar_src.read(window=window)
        label_patch = label_src.read(1, window=window)

        valid_mask = ~np.isnan(sar_patch[0])
        valid_fraction = valid_mask.sum() / valid_mask.size

        if valid_fraction < VALID_THRESHOLD:
            continue

        sar_patch = np.nan_to_num(sar_patch, nan=0.0).astype(np.float16)
        label_patch = np.nan_to_num(label_patch, nan=0.0).astype(np.uint8)

        has_flood = bool((label_patch == 1).any())

        sar_out_path = os.path.join(OUTPUT_DIR, f"sar_{patch_id:05d}.npy")
        label_out_path = os.path.join(OUTPUT_DIR, f"label_{patch_id:05d}.npy")
        np.save(sar_out_path, sar_patch)
        np.save(label_out_path, label_patch)

        manifest.append({
            "patch_id": patch_id,
            "row_off": row_off,
            "col_off": col_off,
            "valid_fraction": float(valid_fraction),
            "has_flood": has_flood,
            "sar_path": sar_out_path,
            "label_path": label_out_path,
        })
        patch_id += 1

sar_src.close()
label_src.close()

print(f"Total patches kept: {patch_id} out of {total_patches} candidate grid cells")

In [ ]:
# Aim: save the patch manifest as a CSV and summarize class balance across kept patches

import pandas as pd

manifest_df = pd.DataFrame(manifest)
manifest_csv_path = os.path.join(OUTPUT_DIR, "manifest.csv")
manifest_df.to_csv(manifest_csv_path, index=False)

flood_patch_count = manifest_df["has_flood"].sum()
total_kept = len(manifest_df)

print(f"Manifest saved to {manifest_csv_path}")
print(f"Patches containing at least one flood pixel: {flood_patch_count} out of {total_kept} ({100*flood_patch_count/total_kept:.2f}%)")
print(f"Patches with no flood pixels at all: {total_kept - flood_patch_count}")

In [ ]:
# Aim: recompute manifest with flood pixel density per patch instead of a binary flag, since binary flag proved uninformative at this patch size

density_records = []
for record in manifest:
    label_patch = np.load(record["label_path"])
    flood_fraction = float((label_patch == 1).sum() / label_patch.size)
    density_records.append(flood_fraction)

manifest_df["flood_fraction"] = density_records
manifest_df.to_csv(manifest_csv_path, index=False)

print(manifest_df["flood_fraction"].describe())
print()
print("Patches with flood_fraction < 0.01:", (manifest_df["flood_fraction"] < 0.01).sum())
print("Patches with flood_fraction between 0.01 and 0.20:", ((manifest_df["flood_fraction"] >= 0.01) & (manifest_df["flood_fraction"] < 0.20)).sum())
print("Patches with flood_fraction >= 0.20:", (manifest_df["flood_fraction"] >= 0.20).sum())

In [ ]:
# Aim: visually verify SAR band alignment and label correspondence on a sample of patches before finalizing the dataset

import matplotlib.pyplot as plt

sample_ids = manifest_df.sort_values("flood_fraction", ascending=False).head(3)["patch_id"].tolist()

fig, axes = plt.subplots(len(sample_ids), 3, figsize=(12, 4 * len(sample_ids)))

for i, pid in enumerate(sample_ids):
    record = manifest_df[manifest_df["patch_id"] == pid].iloc[0]
    sar_patch = np.load(record["sar_path"])
    label_patch = np.load(record["label_path"])

    axes[i, 0].imshow(sar_patch[0], cmap="gray")
    axes[i, 0].set_title(f"patch {pid}: VV_post")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(sar_patch[4], cmap="gray")
    axes[i, 1].set_title(f"patch {pid}: log_ratio")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(label_patch, cmap="gray")
    axes[i, 2].set_title(f"patch {pid}: label")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/patches/phase1_sample_patches.png", dpi=100)
plt.show()

In [ ]:
# Aim: apply a majority filter to a sample label patch to check whether coherent flood structure emerges under the noise

from scipy import ndimage

sample_id = 1239
record = manifest_df[manifest_df["patch_id"] == sample_id].iloc[0]
label_patch = np.load(record["label_path"])

filtered = ndimage.median_filter(label_patch, size=5)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(label_patch, cmap="gray")
axes[0].set_title(f"patch {sample_id}: original label")
axes[0].axis("off")

axes[1].imshow(filtered, cmap="gray")
axes[1].set_title(f"patch {sample_id}: median filtered (5x5)")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/patches/label_noise_check.png", dpi=100)
plt.show()

In [ ]:
# Aim: apply median filtering to all label patches to remove speckle noise before finalizing the training dataset

from scipy import ndimage

FILTER_SIZE = 5
filtered_dir = os.path.join(OUTPUT_DIR, "labels_filtered")
os.makedirs(filtered_dir, exist_ok=True)

filtered_flood_fractions = []

for record in manifest:
    label_patch = np.load(record["label_path"])
    filtered_patch = ndimage.median_filter(label_patch, size=FILTER_SIZE).astype(np.uint8)

    filtered_path = os.path.join(filtered_dir, f"label_filtered_{record['patch_id']:05d}.npy")
    np.save(filtered_path, filtered_patch)

    record["label_filtered_path"] = filtered_path
    filtered_flood_fractions.append(float((filtered_patch == 1).sum() / filtered_patch.size))

manifest_df["label_filtered_path"] = [r["label_filtered_path"] for r in manifest]
manifest_df["flood_fraction_filtered"] = filtered_flood_fractions
manifest_df.to_csv(manifest_csv_path, index=False)

print(manifest_df[["flood_fraction", "flood_fraction_filtered"]].describe())

In [ ]:
# Aim: visually compare original versus filtered labels across several density tiers to check whether real flood structure was eroded

sample_ids = [
    manifest_df.sort_values("flood_fraction", ascending=False).iloc[0]["patch_id"],
    manifest_df.iloc[(manifest_df["flood_fraction"] - manifest_df["flood_fraction"].median()).abs().argsort()[:1]]["patch_id"].values[0],
    manifest_df.sort_values("flood_fraction", ascending=True).iloc[0]["patch_id"],
]

fig, axes = plt.subplots(len(sample_ids), 2, figsize=(9, 4 * len(sample_ids)))

for i, pid in enumerate(sample_ids):
    record = manifest_df[manifest_df["patch_id"] == pid].iloc[0]
    original = np.load(record["label_path"])
    filtered = np.load(record["label_filtered_path"])

    axes[i, 0].imshow(original, cmap="gray")
    axes[i, 0].set_title(f"patch {pid}: original")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(filtered, cmap="gray")
    axes[i, 1].set_title(f"patch {pid}: filtered")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/patches/label_filter_comparison.png", dpi=100)
plt.show()

In [ ]:
# Aim: compare 3x3 median filter and morphological opening against the original on the same low-density and high-density samples

from scipy import ndimage

sample_ids = [3474, 1752, 284]
methods = {
    "original": lambda x: x,
    "median_3x3": lambda x: ndimage.median_filter(x, size=3),
    "opening_2x2": lambda x: ndimage.binary_opening(x, structure=np.ones((2, 2))).astype(np.uint8),
}

fig, axes = plt.subplots(len(sample_ids), len(methods), figsize=(4 * len(methods), 4 * len(sample_ids)))

for i, pid in enumerate(sample_ids):
    record = manifest_df[manifest_df["patch_id"] == pid].iloc[0]
    original = np.load(record["label_path"])
    for j, (method_name, method_fn) in enumerate(methods.items()):
        result = method_fn(original)
        axes[i, j].imshow(result, cmap="gray")
        axes[i, j].set_title(f"patch {pid}: {method_name}")
        axes[i, j].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/patches/label_filter_comparison_v2.png", dpi=100)
plt.show()

In [ ]:
# Aim: apply 2x2 morphological opening to all label patches and compare resulting flood fraction distribution against the original

from scipy import ndimage

opened_dir = os.path.join(OUTPUT_DIR, "labels_opened")
os.makedirs(opened_dir, exist_ok=True)

opened_flood_fractions = []
structure = np.ones((2, 2))

for record in manifest:
    label_patch = np.load(record["label_path"])
    opened_patch = ndimage.binary_opening(label_patch, structure=structure).astype(np.uint8)

    opened_path = os.path.join(opened_dir, f"label_opened_{record['patch_id']:05d}.npy")
    np.save(opened_path, opened_patch)

    record["label_opened_path"] = opened_path
    opened_flood_fractions.append(float((opened_patch == 1).sum() / opened_patch.size))

manifest_df["label_opened_path"] = [r["label_opened_path"] for r in manifest]
manifest_df["flood_fraction_opened"] = opened_flood_fractions
manifest_df.to_csv(manifest_csv_path, index=False)

print(manifest_df[["flood_fraction", "flood_fraction_filtered", "flood_fraction_opened"]].describe())
print()
print("Patches with zero flood area after opening:", (manifest_df["flood_fraction_opened"] == 0).sum())
print("Patches with zero flood area after median filter:", (manifest_df["flood_fraction_filtered"] == 0).sum())

In [ ]:
# Aim: finalize manifest with opened labels as ground truth and drop legacy filtered/original label columns from training reference

manifest_df["label_final_path"] = manifest_df["label_opened_path"]
manifest_df["flood_fraction_final"] = manifest_df["flood_fraction_opened"]
manifest_df.to_csv(manifest_csv_path, index=False)

print(manifest_df[["patch_id", "sar_path", "label_final_path", "flood_fraction_final"]].head())
print(f"Final manifest rows: {len(manifest_df)}")

In [ ]:
# Aim: package patch dataset and manifest for Kaggle output, confirm final storage footprint stays within budget

import subprocess

result = subprocess.run(["du", "-sh", OUTPUT_DIR], capture_output=True, text=True)
print(result.stdout)

result_json = subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True)
print(result_json.stdout)